# 🏛️ Kerala Government Orders NER Model (XLM-RoBERTa)

Welcome to the **Named Entity Recognition (NER)** pipeline! This notebook implements the complete end-to-end pipeline:
1. **Auto-Labeling (Weak Supervision)**: Parse all 222 markdown files to automatically extract the issuing Department, GO Number, and Date, generating separate `train.json` and `val.json` splits at the document-level to ensure zero data leakage. Refined rules are used to avoid weak supervision noise from document abstracts.
2. **Data Preparation**: Tokenize the text using XLM-RoBERTa's tokenizer and align the character spans of our entities to the subword tokens using BIO tagging.
3. **Model Fine-Tuning**: Unfreeze the top 1 encoder layer of XLM-RoBERTa and train on CPU for 20 epochs, saving the trained model.
4. **Interactive Extraction**: Extract entities from unseen text using our deep-learning pure NER model without any post-processing regex rules.

## 📂 Stage 1: Automated Dataset Labeling
This cell reads every markdown file in the folder, parses the header lines, and extracts the Department, GO Number, and Date using robust pattern matching to compile the training dataset `train.json` in less than a second! It handles various types of government orders (Routine, Printed, Manuscript) and various formatting layouts.

In [ ]:
import os
import json
import re
import random

def auto_label_dataset(directory='.'):
    # Gather files
    md_files = sorted([f for f in os.listdir(directory) if f.endswith('.md')])
    md_files = [f for f in md_files if not f.startswith('Project_Report') and f != 'README.md']
    
    # Extraction patterns (used only for labeling the dataset)
    go_pattern = r'G\.?\s*O\.?\s*(?:\(?\s*(?:Rt|Ms|P)\.?\s*\)?\s*)?(?:No\.|No:|No)\s*[:.]?\s*[\d/A-Za-z.-]+'
    date_pattern = r'\b(\d{1,2}[./-]\d{1,2}[./-]\d{4})\b'
    
    random.seed(42)
    shuffled_files = md_files.copy()
    random.shuffle(shuffled_files)
    
    # Split files 85% train, 15% val
    split_idx = int(len(shuffled_files) * 0.85)
    train_files = shuffled_files[:split_idx]
    val_files = shuffled_files[split_idx:]
    
    def process_files(files_list):
        data = []
        for f_name in files_list:
            path = os.path.join(directory, f_name)
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            lines = content.split('\n')
            
            dept_text = None
            for line in lines[:30]:
                if len(line) > 80:
                    continue
                if any(kw in line.upper() for kw in ['REPORT', 'COMMISSION', 'WP(C)', 'JUDGEMENT', 'COMPLIED']):
                    continue
                if 'DEPARTMENT' in line.upper() and ('FINANCE' in line.upper() or 'GOVERNMENT OF KERALA' in line.upper()):
                    cleaned = line.replace('#', '').replace('*', '').replace('_', '').strip()
                    m = re.search(r'(FINANCE\s*(?:\([^)]*\)\s*)?DEPARTMENT)', cleaned, re.IGNORECASE)
                    if m:
                        dept_text = m.group(1).strip()
                        break
                    elif cleaned.upper().startswith('FINANCE'):
                        dept_text = cleaned
                        break
                        
            go_text = None
            go_line_idx = -1
            for idx, line in enumerate(lines[:30]):
                cleaned = line.replace('#', '').replace('*', '').replace('_', '').strip()
                if any(x in cleaned.upper() for x in ['READ', 'LETTER', 'U.O. NOTE']):
                    continue
                m = re.search(go_pattern, cleaned, re.IGNORECASE)
                if m:
                    go_text = m.group(0).strip()
                    go_line_idx = idx
                    break
                    
            date_text = None
            if go_line_idx != -1:
                m = re.search(date_pattern, lines[go_line_idx])
                if m:
                    date_text = m.group(1)
            
            if not date_text and go_line_idx != -1:
                for offset in [-1, 1, 2]:
                    idx = go_line_idx + offset
                    if 0 <= idx < len(lines[:30]):
                        if any(x in lines[idx].upper() for x in ['READ', 'LETTER', 'U.O.', 'WP(C)', 'JUDGEMENT', 'REFERENCE']):
                            continue
                        m = re.search(date_pattern, lines[idx])
                        if m:
                            date_text = m.group(1)
                            break
                            
            if not date_text:
                for line in lines[:30]:
                    if any(x in line.upper() for x in ['READ', 'LETTER', 'U.O.', 'WP(C)', 'JUDGEMENT', 'REFERENCE']):
                        continue
                    m = re.search(date_pattern, line)
                    if m:
                        date_text = m.group(1)
                        break
                            
            if not (dept_text or go_text or date_text):
                continue
                
            pos_lines = []
            neg_lines = []
            
            for line in lines[:30]:
                line_clean = line.replace('#', ' ').replace('*', ' ').replace('_', ' ').replace('\\', ' ')
                line_clean = re.sub(r'\s+', ' ', line_clean).strip()
                if not line_clean or len(line_clean) < 4:
                    continue
                    
                entities = []
                is_reference_line = any(x in line_clean.upper() for x in ['READ', 'LETTER', 'U.O.', 'WP(C)', 'JUDGEMENT', 'REFERENCE'])
                
                if dept_text:
                    m = re.search(re.escape(dept_text), line_clean, re.IGNORECASE)
                    if m:
                        entities.append([m.group(0), 'DEPARTMENT'])
                if go_text and not is_reference_line:
                    m = re.search(re.escape(go_text), line_clean, re.IGNORECASE)
                    if m:
                        entities.append([m.group(0), 'GO_NUMBER'])
                if date_text and not is_reference_line:
                    m = re.search(re.escape(date_text), line_clean, re.IGNORECASE)
                    if m:
                        entities.append([m.group(0), 'DATE'])
                        
                if entities:
                    pos_lines.append({
                        'text': line_clean,
                        'entities': entities
                    })
                else:
                    if len(line_clean) > 8:
                        neg_lines.append({
                            'text': line_clean,
                            'entities': []
                        })
                        
            data.extend(pos_lines)
            if neg_lines:
                data.extend(random.sample(neg_lines, min(len(neg_lines), 2)))
                
        return data
        
    train_data = process_files(train_files)
    val_data = process_files(val_files)
    
    with open('train.json', 'w', encoding='utf-8') as f:
        json.dump(train_data, f, indent=4)
    with open('val.json', 'w', encoding='utf-8') as f:
        json.dump(val_data, f, indent=4)
        
    print(f'Generated dataset splits with zero document leakage:')
    print(f'Training split: {len(train_data)} lines (from {len(train_files)} files)')
    print(f'Validation split: {len(val_data)} lines (from {len(val_files)} files)')
    return train_data, val_data

train_examples, val_examples = auto_label_dataset()


## ⚙️ Stage 2: Imports and System Configuration
We import PyTorch, Hugging Face `transformers`, `datasets`, and check if we are training on CPU or GPU.

In [2]:
import os
import json
import torch
import random
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on Device:', device)

C:\Users\adnan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_matrix, issparse


PyTorch Version: 2.11.0+cpu
CUDA Available: False
Training on Device: cpu


## 📂 Stage 3: Load and Split Dataset
We load the generated `train.json` file and split it into training (85%) and validation (15%) subsets.

In [3]:
import json

with open('train.json', 'r', encoding='utf-8') as f:
    train_examples = json.load(f)
with open('val.json', 'r', encoding='utf-8') as f:
    val_examples = json.load(f)

print(f'Train size: {len(train_examples)} | Validation size: {len(val_examples)}')
print('\nSample train example:')
print(json.dumps(train_examples[0], indent=2))


Train size: 710 | Validation size: 127

Sample train example:
{
  "text": "FINANCE (PENSION-B) DEPARTMENT",
  "entities": [
    [
      "FINANCE (PENSION-B) DEPARTMENT",
      "DEPARTMENT"
    ]
  ]
}


## 🏷️ Stage 4: BIO Tags & Mappings
To train a token classifier, we represent our entities using standard **BIO (Beginning, Inside, Outside) Tagging** format.

In [4]:
label_list = [
    'O',
    'B-DEPARTMENT',
    'I-DEPARTMENT',
    'B-GO_NUMBER',
    'I-GO_NUMBER',
    'B-DATE',
    'I-DATE'
]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print('Label list:', label_list)
print('Label to ID map:', label2id)

Label list: ['O', 'B-DEPARTMENT', 'I-DEPARTMENT', 'B-GO_NUMBER', 'I-GO_NUMBER', 'B-DATE', 'I-DATE']
Label to ID map: {'O': 0, 'B-DEPARTMENT': 1, 'I-DEPARTMENT': 2, 'B-GO_NUMBER': 3, 'I-GO_NUMBER': 4, 'B-DATE': 5, 'I-DATE': 6}


## 🔤 Stage 5: Initialize XLM-RoBERTa Tokenizer and Align Labels
We load the `xlm-roberta-base` tokenizer from Hugging Face and align character-level annotations to the subword tokens. Sequence length is optimized to 64 for rapid CPU training.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

model_checkpoint = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def align_and_tokenize(examples_list):
    tokenized_inputs = {
        'input_ids': [],
        'attention_mask': [],
        'labels': []
    }
    
    for ex in examples_list:
        text = ex['text']
        entities = ex['entities']
        
        char_labels = ['O'] * len(text)
        for ent_text, ent_label in entities:
            start = text.find(ent_text)
            if start == -1:
                continue
            end = start + len(ent_text)
            
            char_labels[start] = f'B-{ent_label}'
            for i in range(start + 1, end):
                char_labels[i] = f'I-{ent_label}'
                
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=32,
            return_offsets_mapping=True,
            padding=False
        )
        
        labels = []
        offsets = tokenized['offset_mapping']
        
        for idx, (start, end) in enumerate(offsets):
            if start == 0 and end == 0:
                labels.append(-100)
            else: 
                token_char_labels = char_labels[start:end]
                non_o_labels = [l for l in token_char_labels if l != 'O']
                
                if non_o_labels:
                    labels.append(label2id[non_o_labels[0]])
                else:
                    labels.append(label2id['O'])
                    
        tokenized_inputs['input_ids'].append(tokenized['input_ids'])
        tokenized_inputs['attention_mask'].append(tokenized['attention_mask'])
        tokenized_inputs['labels'].append(labels)
        
    return Dataset.from_dict(tokenized_inputs)

train_dataset = align_and_tokenize(train_examples)
val_dataset = align_and_tokenize(val_examples)
print('Processed train & validation datasets!')


Processed train & validation datasets!


## 🧠 Stage 6: Instantiate XLM-RoBERTa Model & Freeze Backbone
We load `xlm-roberta-base` with a sequence classification head configured with our 7 target classes. To speed up fine-tuning on CPU to ~30 seconds, we freeze all parameters of the RoBERTa backbone.

In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Freeze base parameters except the top 1 encoder layer for fast CPU domain adaptation
for param in model.roberta.parameters():
    param.requires_grad = False

for layer in model.roberta.encoder.layer[-1:]:
    for param in layer.parameters():
        param.requires_grad = True

print('Model loaded successfully, top 1 encoder layer unfrozen!')


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully, top 1 encoder layer unfrozen!


## 🏋️‍♂️ Stage 7: Fine-Tuning Execution
We specify hyper-parameters and initialize the Hugging Face `Trainer`. We will train the classification head for 3 epochs on CPU, which completes in about 30 seconds.

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    
    flat_preds = preds.flatten()
    flat_labels = labels.flatten()
    
    mask = flat_labels != -100
    flat_preds = flat_preds[mask]
    flat_labels = flat_labels[mask]
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, 
        flat_preds, 
        average='micro',
        labels=[1, 2, 3, 4, 5, 6],
        zero_division=0
    )
    
    accuracy = np.mean(flat_preds == flat_labels)
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy
    }

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='no',  # Disallowed intermediate checkpoints to prevent CPU MemoryError
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    logging_steps=5,
    use_cpu=True,
    report_to='none'
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('Starting classifier head fine-tuning on CPU (20 epochs)...')
trainer.train()


Starting classifier head fine-tuning on CPU (20 epochs)...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.253600,0.125708,0.995331,0.931818,0.962528,0.962477
2,0.032100,0.031016,0.996479,0.989510,0.992982,0.992767
3,0.108500,0.022430,0.994760,0.995629,0.995194,0.995027
4,0.022100,0.014456,0.994764,0.996503,0.995633,0.995479
5,0.039200,0.018694,0.999118,0.990385,0.994732,0.994575
6,0.019900,0.011064,0.996507,0.997378,0.996942,0.996835
7,0.008900,0.033037,0.999116,0.987762,0.993407,0.993219
8,0.012300,0.009374,0.996507,0.997378,0.996942,0.996835
9,0.006500,0.012865,1.000000,0.993881,0.996931,0.996835
10,0.006700,0.008616,1.000000,0.996503,0.998249,0.998192


TrainOutput(global_step=900, training_loss=0.05324156190165215, metrics={'train_runtime': 1135.2917, 'train_samples_per_second': 12.508, 'train_steps_per_second': 0.793, 'total_flos': 229291123769268.0, 'train_loss': 0.05324156190165215, 'epoch': 20.0})

## 📊 Stage 8: Evaluate and Save Model
Let's measure performance on the validation set and save the model weights and tokenizer to disk.

In [8]:
eval_results = trainer.evaluate()
print('Final model evaluation results:')
for key, value in eval_results.items():
    print(f'{key}: {value}')

model.save_pretrained('./fine_tuned_model_v3')
tokenizer.save_pretrained('./fine_tuned_model_v3')
print('Model and tokenizer saved successfully to ./fine_tuned_model_v3!')


Final model evaluation results:
eval_loss: 0.009356926195323467
eval_precision: 0.9965065502183406
eval_recall: 0.9973776223776224
eval_f1: 0.9969418960244648
eval_accuracy: 0.9968354430379747
eval_runtime: 6.2857
eval_samples_per_second: 20.205
eval_steps_per_second: 1.273
epoch: 20.0


Model and tokenizer saved successfully to ./fine_tuned_model_v2!


## 🚀 Stage 9: Pure Deep Learning Extractor
We build the extractor engine. It processes the document's header block using the fine-tuned XLM-RoBERTa model entirely, without any hybrid extraction, regex fallbacks, heuristic corrections, or rule-based prediction logic.

In [ ]:
class GovernmentOrderExtractor:
    def __init__(self, model_dir):
        from transformers import AutoTokenizer, AutoModelForTokenClassification
        print("* Loading trained XLM-RoBERTa model...")
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to('cpu').eval()
        print("* Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        print("* Model and tokenizer loaded successfully.\n* Extractor initialized successfully.")

    def extract(self, text):
        import re, torch
        lines = [l.strip() for l in text.split('\n')[:30] if l.strip()]
        res = {}
        for line in lines:
            line = re.sub(r'\s+', ' ', line.replace('#', ' ').replace('*', ' ').replace('_', ' ').replace('\\', ' ')).strip()
            if len(line) < 4: continue
            inputs = self.tokenizer(line, return_tensors='pt', return_offsets_mapping=True, truncation=True, max_length=64)
            with torch.no_grad():
                preds = torch.argmax(self.model(inputs['input_ids']).logits, dim=-1)[0].numpy()
            offsets = inputs['offset_mapping'][0].numpy()
            
            curr, start, end = None, -1, -1
            entities = {'DEPARTMENT': [], 'GO_NUMBER': [], 'DATE': []}
            
            for idx, pred in enumerate(preds):
                label = self.model.config.id2label[pred]
                s, e = offsets[idx]
                if s == 0 and e == 0: continue
                if label.startswith('B-'):
                    if curr: entities[curr].append(line[start:end].strip())
                    curr, start, end = label[2:], s, e
                elif label.startswith('I-') and curr == label[2:]:
                    end = e
                elif curr:
                    entities[curr].append(line[start:end].strip())
                    curr = None
            if curr: entities[curr].append(line[start:end].strip())
            
            for k in entities:
                key = k.lower() if k != 'DEPARTMENT' else 'department_name'
                if entities[k] and key not in res:
                    val = entities[k][0].strip()
                    if val and val.lower() not in ('none', 'null'):
                        res[key] = val
        return res


## 📂 Stage 10: Model Inference
Upload a markdown (`.md`) file below to run Named Entity Recognition using the trained XLM-RoBERTa model.

In [ ]:
import json
import os

# Prompt the user to enter the markdown file path
filepath = input("Enter the path to the markdown (.md) file: ").strip()

if os.path.exists(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()

    # Initialize the extractor (triggers loading progress messages) and predict
    extractor = GovernmentOrderExtractor('./fine_tuned_model_v3')
    results = extractor.extract(text)

    print("\nPrediction Results:")
    print(json.dumps(results, indent=4))
else:
    print(f"Error: File '{filepath}' does not exist.")


In [43]:
import os
import pandas as pd

test1_dir = './test1'
model_dir = './fine_tuned_model_v3'

# Initialize extractor using the trained NER model
extractor = GovernmentOrderExtractor(model_dir)

results = []
files = sorted([f for f in os.listdir(test1_dir) if f.endswith('.md')])

for filename in files:
    filepath = os.path.join(test1_dir, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        
    # Run the trained NER model on the content
    predictions = extractor.extract(text)
    
    # Extract only the predicted G.O. Number
    predicted_go = predictions.get('go_number', None)
    
    results.append({
        'Filename': filename,
        'Predicted GO Number': predicted_go
    })

# Display the results in a clear tabular format
df = pd.DataFrame(results)
df


* Loading trained XLM-RoBERTa model...
* Loading tokenizer...
* Model and tokenizer loaded successfully.
* Extractor initialized successfully.


,Filename,Predicted GO Number
0,GO(P)No57-2026-FinDated21-03-2026_93.md,G.O.(P)No.57/2026/FIN
1,GO(P)No63-2026-FinDated07-05-2026_83.md,G.O.(P)No.63/2026/FIN
2,GO(Rt)No2970-2026-FinDated18-03-2026_96.md,G.O.(Rt) No.2970/2026/FIN
3,GO(Rt)No3175-2026-FinDated21-03-2026_96.md,GO (Rt) No.3175/2026/Fin.
4,GO(Rt)No3200-2026-FinDated21-03-2026_99.md,G.O.(Rt)No.3200/2026/FIN
5,GO(Rt)No3249-2026-FinDated23-03-2026_42.md,G.O.(Rt)No.3249/2026/FIN
6,GO(Rt)No3280-2026-FinDated23-03-2026_99.md,G.O.(Rt)No.3280/2026/FIN
7,GO(Rt)No3290-2026-FinDated23-03-2026_42.md,G.O.(Rt)No.3290/2026/FIN
8,GO(Rt)No3299-2026-FinDated23-03-2026_72.md,G.O.(Rt)No.3299/2026/FIN
9,GO(Rt)No3401-2026-FinDated25-03-2026_99.md,G.O.(Rt)No.3401/2026/FIN


In [44]:
import os
import re
import pandas as pd

test1_dir = './test1'
model_dir = './fine_tuned_model_v3'

# Regex pattern (ground truth reference)
go_pattern = r'G\.?\s*O\.?\s*(?:\(?\s*(?:Rt|Ms|P)\.?\s*\)?\s*)?(?:No\.|No:|No)\s*[:.]?\s*[\d/A-Za-z.-]+'

# Initialize extractor using the trained NER model
extractor = GovernmentOrderExtractor(model_dir)

files = sorted([f for f in os.listdir(test1_dir) if f.endswith('.md')])

comparison_data = []
tp, tn, fp, fn = 0, 0, 0, 0

for filename in files:
    filepath = os.path.join(test1_dir, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        
    lines = content.split('\n')
    
    # 1. Regex Extraction (Reference Ground Truth)
    ref_go = None
    for line in lines[:30]:
        cleaned_line = line.replace('#', '').replace('*', '').replace('_', '').strip()
        if any(x in cleaned_line.upper() for x in ['READ', 'LETTER', 'U.O. NOTE', 'WP(C)', 'JUDGEMENT', 'REFERENCE']):
            continue
        m = re.search(go_pattern, cleaned_line, re.IGNORECASE)
        if m:
            ref_go = m.group(0).strip()
            break
            
    # 2. NER Model Prediction
    predictions = extractor.extract(content)
    pred_go = predictions.get('go_number', None)
    
    # 3. Compare
    is_match = (ref_go == pred_go)
    
    # Update Confusion Matrix counters
    if ref_go is not None and pred_go is not None:
        if is_match: tp += 1
        else: fp += 1
    elif ref_go is None and pred_go is None:
        tn += 1
    elif ref_go is None and pred_go is not None:
        fp += 1
    elif ref_go is not None and pred_go is None:
        fn += 1
        
    comparison_data.append({
        'Filename': filename,
        'Regex GO Number': ref_go,
        'Model Predicted GO Number': pred_go,
        'Match': is_match
    })

# Convert to DataFrame
df_comp = pd.DataFrame(comparison_data)

# Calculate metrics
total = len(files)
correct = tp + tn
incorrect = fp + fn

accuracy = correct / total if total > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Display evaluation metrics
print("==================================================")
print("     NER Model vs Regex Extraction Evaluation      ")
print("==================================================")
print(f"Total Files Analyzed:   {total}")
print(f"Correct Predictions:    {correct}")
print(f"Incorrect Predictions:  {incorrect}")
print(f"Accuracy:               {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"Precision:              {precision:.4f}")
print(f"Recall:                 {recall:.4f}")
print(f"F1 Score:               {f1:.4f}")
print("--------------------------------------------------")
print("Confusion Summary:")
print(f"  - True Positives (TP):  {tp}")
print(f"  - True Negatives (TN):  {tn}")
print(f"  - False Positives (FP): {fp}")
print(f"  - False Negatives (FN): {fn}")
print("==================================================\n")

# Display comparison table
pd.set_option('display.max_rows', 50)
df_comp


* Loading trained XLM-RoBERTa model...
* Loading tokenizer...
* Model and tokenizer loaded successfully.
* Extractor initialized successfully.
     NER Model vs Regex Extraction Evaluation      
Total Files Analyzed:   30
Correct Predictions:    30
Incorrect Predictions:  0
Accuracy:               1.0000 (100.0%)
Precision:              1.0000
Recall:                 1.0000
F1 Score:               1.0000
--------------------------------------------------
Confusion Summary:
  - True Positives (TP):  30
  - True Negatives (TN):  0
  - False Positives (FP): 0
  - False Negatives (FN): 0



,Filename,Regex GO Number,Model Predicted GO Number,Match
0,GO(P)No57-2026-FinDated21-03-2026_93.md,G.O.(P)No.57/2026/FIN,G.O.(P)No.57/2026/FIN,True
1,GO(P)No63-2026-FinDated07-05-2026_83.md,G.O.(P)No.63/2026/FIN,G.O.(P)No.63/2026/FIN,True
2,GO(Rt)No2970-2026-FinDated18-03-2026_96.md,G.O.(Rt) No.2970/2026/FIN,G.O.(Rt) No.2970/2026/FIN,True
3,GO(Rt)No3175-2026-FinDated21-03-2026_96.md,GO (Rt) No.3175/2026/Fin.,GO (Rt) No.3175/2026/Fin.,True
4,GO(Rt)No3200-2026-FinDated21-03-2026_99.md,G.O.(Rt)No.3200/2026/FIN,G.O.(Rt)No.3200/2026/FIN,True
5,GO(Rt)No3249-2026-FinDated23-03-2026_42.md,G.O.(Rt)No.3249/2026/FIN,G.O.(Rt)No.3249/2026/FIN,True
6,GO(Rt)No3280-2026-FinDated23-03-2026_99.md,G.O.(Rt)No.3280/2026/FIN,G.O.(Rt)No.3280/2026/FIN,True
7,GO(Rt)No3290-2026-FinDated23-03-2026_42.md,G.O.(Rt)No.3290/2026/FIN,G.O.(Rt)No.3290/2026/FIN,True
8,GO(Rt)No3299-2026-FinDated23-03-2026_72.md,G.O.(Rt)No.3299/2026/FIN,G.O.(Rt)No.3299/2026/FIN,True
9,GO(Rt)No3401-2026-FinDated25-03-2026_99.md,G.O.(Rt)No.3401/2026/FIN,G.O.(Rt)No.3401/2026/FIN,True
